# EDA and Quality Checks

In [ ]:
import sys
import geopandas as gpd
from pathlib import Path
from src.config import PROCESSED_DIR
import matplotlib.pyplot as plt
import seaborn as sns

cwd = Path.cwd()
project_root = cwd.parent if cwd.name == 'notebooks' else cwd
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))



## Load processed files

In [ ]:
# Define file paths
ev_parquet_path = PROCESSED_DIR / "ev_stations.parquet"
gas_parquet_path = PROCESSED_DIR / "gas_stations.parquet"

# Load GeoParquet files
print("Loading GeoParquet files...")
ev_gdf = gpd.read_parquet(ev_parquet_path)
gas_gdf = gpd.read_parquet(gas_parquet_path)

## Schema Inspection

In [ ]:
# Display data types
print("EV Data Types: ")
print(f"{ev_gdf.dtypes}")
print("Gas Data Types: ")
print(f"{gas_gdf.dtypes}")

# Check if required columns exist
required_ev_cols = [
    'id', 'station_name', 'latitude', 'longitude', 'access_code',
    'level1_ports', 'level2_ports', 'dcfc_ports', 'total_ev_ports', 'has_dcfc',
]
required_gas_cols = [
    'station_id', 'name', 'address', 'city', 'zip', 'source', 'source_snapshot_date', 'geometry',
]
missing_ev_cols = [col for col in required_ev_cols if col not in ev_gdf.columns]
missing_gas_cols = [col for col in required_gas_cols if col not in gas_gdf.columns]
assert not missing_ev_cols, f"Missing {missing_ev_cols}"
assert not missing_gas_cols, f"Missing {missing_gas_cols}"

# Confirm if geometry colummns are active
print("EV geometry column:", ev_gdf.geometry.name)
print("Gas geometry column:", gas_gdf.geometry.name)

print("EV geometry dtype:", ev_gdf.geometry.dtype)
print("Gas geometry dtype:", gas_gdf.geometry.dtype)

print("EV null geometries:", ev_gdf.geometry.isna().sum())
print("Gas null geometries:", gas_gdf.geometry.isna().sum())

## EV Inspection

In [ ]:
# Network Counts
network_counts = ev_gdf['ev_network'].value_counts(dropna=False)
print(network_counts)

# Status Count
status_counts = ev_gdf['status_code'].value_counts(dropna=False)
print(status_counts)

# Summarize Port Counts
port_cols = ['level2_ports', 'dcfc_ports', 'total_ev_ports']
ev_gdf[port_cols].describe()

# Histogram for Port Counts
fig, axes = plt.subplots(2,2, figsize=(8,8))
axes = axes.flatten()
for i,col in enumerate(port_cols):
    axes[i].hist(ev_gdf[col])
    axes[i].set_title('{col}')
plt.tight_layout()
plt.show()




## Gas Inspection

In [ ]:
# Count records by city, ZIP
city_counts = gas_gdf['city'].value_counts(dropna=False)
zip_counts = gas_gdf['zip'].value_counts(dropna=False)
print(city_counts.head(20))
print(zip_counts.head(20))

# Recompute n_raw by reading the latest raw snapshot directly
from src.transform.clean_stations import find_latest_files
_, gas_raw_path = find_latest_files()
gas_raw = gpd.read_file(gas_raw_path)

n_raw = len(gas_raw)
n_clean = len(gas_gdf)
n_removed = n_raw - n_clean

print(f"Raw gas stations:      {n_raw}")
print(f"After dedup/cleaning:  {n_clean}")
print(f"Removed:               {n_removed} ({n_removed / n_raw:.1%})")

# Rows with missing names or addresses
missing_names_addresses = gas_gdf[gas_gdf['name'].isna() | gas_gdf['address'].isna()]
print(missing_names_addresses.head(10))

## Spatial Checks

In [ ]:
# Reproject both layers to EPSG:4326 for plotting
ev_web = ev_gdf.to_crs("EPSG:4326")
gas_web = gas_gdf.to_crs("EPSG:4326")

print(f"EV CRS after projection: {ev_web}")
print(f"Gas CRS after projection {gas_web}")

In [ ]:
# Plot points over California boundary
import geopandas as gpd
from src.config import EXTERNAL_DIR

tracts = gpd.read_file(EXTERNAL_DIR / "california_tracts") 
ca_boundary = tracts.to_crs("EPSG:4326").dissolve()  

fig, ax = plt.subplots(figsize=(8, 10))
ca_boundary.boundary.plot(ax=ax, color='black', linewidth=0.5)
gas_web.plot(ax=ax, markersize=3, color='steelblue', alpha=0.5, label='Gas stations')
ev_web.plot(ax=ax, markersize=5, color='darkorange', alpha=0.7, label='EV stations')
ax.set_title("EV and Gas Stations vs. California Boundary")
ax.legend()
plt.show()

In [ ]:
# Numeric check: any points clearly outside CA bounding box after cleaning?
ev_outliers = ev_web[~ev_web.geometry.x.between(-125, -113) | ~ev_web.geometry.y.between(32.5, 43.0)]
gas_outliers = gas_web[~gas_web.geometry.x.between(-125, -113) | ~gas_web.geometry.y.between(32.5, 43.0)]

print(f"EV points outside CA bbox: {len(ev_outliers)}")
print(f"Gas points outside CA bbox: {len(gas_outliers)}")